# Baseline Model (Milestone 2)

**Goal:** a simple, honest reference score that better models must beat.
**Metric:** ROC-AUC (only 8% default), plus recall and precision for defaulters.
**Setup:** `app_test` has no labels, so 20% of `app_train` is held out as a final test set. Model choice uses stratified cross-validation on the remaining 80%.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

DATA_DIR = Path("data/home-credit-default-risk")
SEED = 42

## 1. Load and clean
The same cleaning function will later be applied to `app_test`.

In [6]:
def clean(df):
    df = df.copy()
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    for col in ["CODE_GENDER", "ORGANIZATION_TYPE"]:
        df[col] = df[col].replace("XNA", np.nan)
    cat_cols = df.select_dtypes(exclude="number").columns
    df[cat_cols] = df[cat_cols].astype(object)  # plain object dtype works best with sklearn
    return df

app_train = clean(pd.read_csv(DATA_DIR / "application_train.csv"))

X = app_train.drop(columns=["TARGET", "SK_ID_CURR"])
y = app_train["TARGET"]
print(X.shape, f"default rate: {y.mean():.2%}")

(307511, 121) default rate: 8.07%


## 2. Hold-out split (stratified)

In [7]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("dev:", X_dev.shape, " test:", X_test.shape)

dev: (246008, 121)  test: (61503, 121)


## 3. Naive baselines
- **Dummy:** predicts the same probability for everyone → AUC 0.5 (no skill).
- **Simple rule:** rank applicants by their average external score (low score = high risk), no training needed.

In [ ]:
dummy = DummyClassifier(strategy="prior").fit(X_dev, y_dev)
print("Dummy AUC:", round(roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1]), 4))

ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
fill_value = X_dev[ext].mean(axis=1).median()            # learned from dev (training data)
rule_score = -X_test[ext].mean(axis=1).fillna(fill_value) # applied to test (scored data)
print("Rule AUC (mean EXT_SOURCE):", round(roc_auc_score(y_test, rule_score), 4))
# axis=1 goes across the row, giving one number per applicant:
#  the average of that person’s three scores.
# roc_auc_score can’t handle NaN. The fill value comes from X_dev, not X_test.
# roc_auc_score needs one number per applicant. 
# The .mean(axis=1) turns three EXT_SOURCE columns into one single score.

Dummy AUC: 0.5
Rule AUC (mean EXT_SOURCE): 0.7165


## 4. Baseline model: logistic regression
Median imputation + scaling for numbers, one-hot encoding for categories. `class_weight="balanced"` gives the 8% defaulters more weight.

In [ ]:
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

scores = cross_val_score(logreg, X_dev, y_dev, cv=cv, scoring="roc_auc", n_jobs=-1)
print("CV AUC:", scores.round(4), "mean:", scores.mean().round(4))

## 5. Final check on the hold-out set

In [ ]:
logreg.fit(X_dev, y_dev)
proba = logreg.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Hold-out AUC:", round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, pred, digits=3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[0])
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax[1])
plt.tight_layout()
plt.show()

## 6. Summary
| Model | AUC |
|---|---|
| Dummy | 0.50 |
| Rule (mean EXT_SOURCE) | _fill in_ |
| Logistic regression (CV) | _fill in_ |
| Logistic regression (hold-out) | _fill in_ |

- Recall for defaulters (class 1): _fill in_, precision: _fill in_.
- **Next (Milestone 3):** add side-table features, try at least two stronger models (e.g. random forest, gradient boosting), tune them with CV, and analyse the errors.